# 29 — Two-Phase Commit (Amazon FAR style)

Atomic commit across multiple participants: a coordinator runs a **prepare** (vote) phase, then a
**commit/abort** decision phase — all-or-nothing. Parts 1-3 build the core (concurrency at Part 3);
Parts 4-5 are stretch tasks (recovery + failure handling, then parallel decision replay). Fill stubs,
run each test cell, peek at the collapsed `ref_` solutions only after trying.

---

## Part 1 — Coordinator & participants

`Participant(vote)` with `prepare() -> bool` (its vote), `commit()`, `abort()` (each sets `state`).
`run_2pc(participants) -> "commit" | "abort"`: prepare everyone; if **all** vote yes, commit all
(return `"commit"`); otherwise abort all (return `"abort"`).

**Lock down:** a single "no" vote aborts the whole transaction; the decision is unanimous-or-abort;
participants end in a consistent state (all committed or all aborted).

In [ ]:
class Participant:
    def __init__(self, vote):
        raise NotImplementedError

    def prepare(self):
        raise NotImplementedError

    def commit(self):
        raise NotImplementedError

    def abort(self):
        raise NotImplementedError


def run_2pc(participants):
    raise NotImplementedError

In [ ]:
# --- Part 1 tests ---
def _t1():
    ps = [Participant(True), Participant(True), Participant(True)]
    assert run_2pc(ps) == "commit" and all(p.state == "committed" for p in ps)
    ps2 = [Participant(True), Participant(False), Participant(True)]
    assert run_2pc(ps2) == "abort" and all(p.state == "aborted" for p in ps2)
    print("Part 1 OK")

_t1()

## Part 2 — Recovery

After a crash, a participant rebuilds its state from its log. `recover_state(log) -> str`:
- a logged `"commit"` → `"committed"`; a logged `"abort"` → `"aborted"`;
- `"prepared"` but no decision → `"in-doubt"` (it voted yes and must ask the coordinator);
- nothing logged → `"aborted"` (presumed abort).

**Discuss:** the **in-doubt / blocking** problem is 2PC's Achilles heel (a participant that prepared
can't unilaterally decide if the coordinator is unreachable); presumed-abort optimization; write-ahead
logging before voting.

In [ ]:
def recover_state(log):
    raise NotImplementedError

In [ ]:
# --- Part 2 tests ---
def _t2():
    assert recover_state(["prepared", "commit"]) == "committed"
    assert recover_state(["prepared", "abort"]) == "aborted"
    assert recover_state(["prepared"]) == "in-doubt"        # crash after voting, before decision
    assert recover_state([]) == "aborted"                   # presumed abort
    print("Part 2 OK")

_t2()

## Part 3 — Concurrent transactions

`ConcurrentCoordinator` runs many independent transactions at once; `run(participants)` returns the
outcome and **tallies** `committed` / `aborted` counts thread-safely.

**The invariant:** 200 transactions (100 designed to commit, 100 to abort) run across threads leave
`committed == 100` and `aborted == 100` — no lost tally updates. **Discuss:** transactions are
independent (parallel), but the shared counters are a read-modify-write needing a lock; in real systems
each participant holds resource locks from prepare until the decision (where deadlocks lurk).

In [ ]:
import threading


class ConcurrentCoordinator:
    def __init__(self):
        raise NotImplementedError

    def run(self, participants):
        raise NotImplementedError

In [ ]:
# --- Part 3 tests ---
def _t3():
    co = ConcurrentCoordinator()

    def worker(should_commit):
        ps = [Participant(should_commit), Participant(True)]
        co.run(ps)

    ts = [threading.Thread(target=worker, args=(i % 2 == 0,)) for i in range(200)]
    for t in ts: t.start()
    for t in ts: t.join()
    assert co.committed == 100 and co.aborted == 100
    print("Part 3 OK")

_t3()

## Part 4 (stretch) — Failure handling

`run_2pc_safe(participants) -> str`: a participant that **fails during prepare** (raises) counts as a
"no" vote → the transaction aborts; abort calls that themselves raise are swallowed so one bad
participant can't wedge the rollback.

**Discuss:** prepare timeouts treated as no-votes, why commit must be retried (not aborted) once the
decision is made, and idempotent commit/abort for safe retries.

In [ ]:
def run_2pc_safe(participants):
    raise NotImplementedError

In [ ]:
# --- Part 4 tests ---
def _t4():
    class Failing:
        def __init__(self):
            self.state = "init"

        def prepare(self):
            raise RuntimeError("crashed during prepare")

        def commit(self):
            self.state = "committed"

        def abort(self):
            self.state = "aborted"

    ps = [Participant(True), Failing(), Participant(True)]
    assert run_2pc_safe(ps) == "abort"                  # a failed prepare aborts the txn
    assert ps[0].state == "aborted" and ps[2].state == "aborted"
    print("Part 4 OK")

_t4()

## Part 5 (stretch) — Parallel decision replay

`parallel_decide(transactions, num_procs) -> list[str]`: given many transactions (each a list of
participant votes), compute each one's outcome in parallel with `ProcessPoolExecutor`; worker is
`tpc_workers.decide` (commit iff all votes are yes).

**Discuss:** the decision is a pure function of the votes (replayable / auditable); transactions are
independent (parallel); this is how you'd re-derive outcomes from a transaction log offline.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
import tpc_workers


def parallel_decide(transactions, num_procs) -> list:
    raise NotImplementedError

In [ ]:
# --- Part 5 tests ---
def _t5():
    txns = [[True, True, True], [True, False], [True, True], [False]]
    assert parallel_decide(txns, 2) == ["commit", "abort", "commit", "abort"]
    print("Part 5 OK")

_t5()

## Likely follow-ups
- Why 2PC blocks if the coordinator fails after prepare; three-phase commit (3PC) and its assumptions.
- Sagas / compensating transactions as a non-blocking alternative; idempotency keys.
- Consensus (Paxos/Raft) for fault-tolerant commit; coordinator high-availability.

---
## Reference solutions (don't peek until you've tried)

In [ ]:
import threading
from concurrent.futures import ProcessPoolExecutor
import tpc_workers


class RefParticipant:
    def __init__(self, vote):
        self.vote = vote
        self.state = "init"

    def prepare(self):
        self.state = "prepared" if self.vote else "abort"
        return self.vote

    def commit(self):
        self.state = "committed"

    def abort(self):
        self.state = "aborted"


def ref_run_2pc(participants):
    votes = [p.prepare() for p in participants]
    if all(votes):
        for p in participants:
            p.commit()
        return "commit"
    for p in participants:
        p.abort()
    return "abort"


def ref_recover_state(log):
    if "commit" in log:
        return "committed"
    if "abort" in log:
        return "aborted"
    if "prepared" in log:
        return "in-doubt"                               # voted yes, decision unknown
    return "aborted"                                    # presumed abort


class RefConcurrentCoordinator:
    def __init__(self):
        self.committed = 0
        self.aborted = 0
        self.lock = threading.Lock()

    def run(self, participants):
        outcome = ref_run_2pc(participants)
        with self.lock:
            if outcome == "commit":
                self.committed += 1
            else:
                self.aborted += 1
        return outcome


def ref_run_2pc_safe(participants):
    votes = []
    for p in participants:
        try:
            votes.append(p.prepare())
        except Exception:                              # a failed prepare = a "no" vote
            votes.append(False)
    if all(votes):
        for p in participants:
            p.commit()
        return "commit"
    for p in participants:
        try:
            p.abort()
        except Exception:                              # don't let one bad rollback wedge the rest
            pass
    return "abort"


def ref_parallel_decide(transactions, num_procs):
    with ProcessPoolExecutor(max_workers=num_procs) as ex:
        return list(ex.map(tpc_workers.decide, transactions))


assert ref_run_2pc([RefParticipant(True), RefParticipant(True)]) == "commit"
assert ref_run_2pc([RefParticipant(True), RefParticipant(False)]) == "abort"
assert ref_recover_state(["prepared"]) == "in-doubt" and ref_recover_state([]) == "aborted"
_co = RefConcurrentCoordinator(); _co.run([RefParticipant(True)]); _co.run([RefParticipant(False)])
assert _co.committed == 1 and _co.aborted == 1
assert ref_parallel_decide([[True, True], [True, False]], 2) == ["commit", "abort"]
print("reference solutions OK")